In [56]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd

In [57]:
sample_submission_df = pd.read_csv("./sample_submission.csv")
train_df = pd.read_csv("./train.csv")
test_df = pd.read_csv("./test.csv")

# Save test IDs for the final submission
test_ids = test_df["id"]

# Separate target and features
y = train_df["health_condition"]
x = train_df.drop(columns=["health_condition", "id"])
x_test = test_df.drop(columns=["id"])

print("Training features:", x.shape)
print("Target:", y.shape)
print("Test features:", x_test.shape)

Training features: (690088, 13)
Target: (690088,)
Test features: (295753, 13)


In [58]:
def preprocess_data(x, x_test):
    x = x.copy()
    x_test = x_test.copy()

    numeric_columns = x.select_dtypes(include="number").columns
    categorical_columns = x.select_dtypes(exclude="number").columns

    numeric_fill_values = x[numeric_columns].median()
    categorical_fill_values = x[categorical_columns].mode().iloc[0]

    x[numeric_columns] = x[numeric_columns].fillna(
        numeric_fill_values
    )

    x[categorical_columns] = x[categorical_columns].fillna(
        categorical_fill_values
    )

    x_test[numeric_columns] = x_test[numeric_columns].fillna(
        numeric_fill_values
    )

    x_test[categorical_columns] = x_test[categorical_columns].fillna(
        categorical_fill_values
    )

    return x, x_test

x_clean, x_test_clean = preprocess_data(x, x_test)

In [59]:
def train_features(x_train_all, x_valid_all, x_train_selected, x_valid_selected, y_train, y_valid):
    # Model using every feature
    model.fit(x_train_all, y_train)
    all_features_score = model.score(x_valid_all, y_valid)

    # Model using selected features
    model.fit(x_train_selected, y_train)
    selected_features_score = model.score(x_valid_selected, y_valid)

    print("All features:", all_features_score)
    print("Selected features:", selected_features_score)

In [60]:
from sklearn.metrics import accuracy_score, classification_report

def training_loop(model, x_train, y_train, x_valid, y_valid):
    model.fit(x_train, y_train)

    valid_predictions = model.predict(x_valid)

    accuracy = accuracy_score(y_valid, valid_predictions)

    print("Validation accuracy:", accuracy)
    print(classification_report(y_valid, valid_predictions))

    return model

In [61]:
def prediction_loop(model, x_test, test_ids):
    predictions = model.predict(x_test)

    submission_df = pd.DataFrame({
        "id": test_ids,
        "health_condition": predictions
    })

    return submission_df

In [62]:
from sklearn.model_selection import train_test_split

x_train, x_valid, y_train, y_valid = train_test_split(
    x_clean,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [63]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_columns = x_train.select_dtypes(include="number").columns
categorical_columns = x_train.select_dtypes(exclude="number").columns

transformer = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_columns),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_columns
    )
])

model = Pipeline([
    ("transformer", transformer),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [64]:
model = training_loop(
    model,
    x_train,
    y_train,
    x_valid,
    y_valid
)

submission_df = prediction_loop(
    model,
    x_test_clean,
    test_ids
)

display(submission_df.head())

submission_df.to_csv(
    "submission.csv",
    index=False
)

print("Submission created successfully!")

Validation accuracy: 0.9480285180193888
              precision    recall  f1-score   support

     at-risk       0.96      0.98      0.97    118512
         fit       0.88      0.73      0.80      7961
   unhealthy       0.89      0.72      0.80     11545

    accuracy                           0.95    138018
   macro avg       0.91      0.81      0.86    138018
weighted avg       0.95      0.95      0.95    138018



,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


Submission created successfully!
